# HTAN — Hyper TransAttUNet
### Bowl — Nuclei Segmentation Experiments
---

## 0. Setup

In [ ]:
import os
import sys
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root : {PROJECT_ROOT}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

---
# 3. Bowl — Nuclei Segmentation

## 3.0 Download Data

```bash
mkdir -p /opt/dlami/nvme/HTAN/data/bowl
cd /opt/dlami/nvme/HTAN/data/bowl
kaggle competitions download -c data-science-bowl-2018
unzip data-science-bowl-2018.zip
```

## 3.1 Verify Data

In [ ]:
from datasets.bowl_dataset import get_split, TRAIN_ROOT

train_ids, val_ids, test_ids = get_split(seed=123)

print(f"Train : {len(train_ids)} images")
print(f"Val   : {len(val_ids)} images")
print(f"Test  : {len(test_ids)} images")
print(f"Total : {len(train_ids)+len(val_ids)+len(test_ids)} images")
# Expected: ~537 / ~67 / ~67

In [ ]:
# Visualize 3 random samples
from datasets.bowl_dataset import merge_masks

samples = random.sample(train_ids, 3)
fig, axes = plt.subplots(3, 2, figsize=(10, 12))
fig.suptitle("Bowl Sample Images", fontsize=14, fontweight="bold")

for i, img_id in enumerate(samples):
    img_path = TRAIN_ROOT / img_id / "images" / f"{img_id}.png"
    mask_dir = TRAIN_ROOT / img_id / "masks"

    img  = Image.open(img_path).convert("RGB")
    mask = merge_masks(mask_dir)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Image: {img_id[:12]}...")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title("Merged mask")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

## 3.2 Sanity Check — Model Forward Pass

Bowl uses **256×256** — same as ISIC.

In [ ]:
from models.transattunet.TransAttUnet import TransAttUNet_R
from models.baselines.unet import UNet
from models.baselines.doubleunet import DoubleUNet
from models.htan.htan import HTAN_1, HTAN_2, HTAN_1_Hres_only

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
dummy  = torch.randn(2, 3, 256, 256).to(DEVICE)

models_to_check = {
    "unet":             UNet(),
    "doubleunet":       DoubleUNet(),
    "transattunet":     TransAttUNet_R(),
    "htan_1_n2":        HTAN_1(expansion_n=2, img_size=256),
    "htan_1_n4":        HTAN_1(expansion_n=4, img_size=256),
    "htan_2_n2":        HTAN_2(expansion_n=2, img_size=256),
    "htan_1_hres_only": HTAN_1_Hres_only(expansion_n=4, img_size=256),
}

print(f"{'Model':<22} {'Params':>12} {'Output':>15} {'Status'}")
print("-" * 60)
for name, model in models_to_check.items():
    try:
        model = model.to(DEVICE).eval()
        with torch.no_grad():
            out = model(dummy)
        n = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"{name:<22} {n:>12,} {str(tuple(out.shape)):>15}   OK")
    except Exception as e:
        print(f"{name:<22} ERROR — {str(e)[:50]}")
    finally:
        del model
        torch.cuda.empty_cache()

## 3.3 Training

In [ ]:
# 3.3.1 TransAttUNet_R (baseline)
!python3 train.py --model transattunet --dataset bowl

In [ ]:
!python3 evaluate.py --model transattunet --dataset bowl

In [ ]:
# 3.3.2 HTAN_2_n2 (best model)
!python3 train.py --model htan_2_n2 --dataset bowl

In [ ]:
!python3 evaluate.py --model htan_2_n2 --dataset bowl

In [ ]:
# 3.3.3 HTAN_1_n2
!python3 train.py --model htan_1_n2 --dataset bowl

In [ ]:
!python3 evaluate.py --model htan_1_n2 --dataset bowl

In [ ]:
# 3.3.4 HTAN_1_n4
!python3 train.py --model htan_1_n4 --dataset bowl

In [ ]:
!python3 evaluate.py --model htan_1_n4 --dataset bowl

In [ ]:
# 3.3.5 HTAN_1_hres_only (ablation)
!python3 train.py --model htan_1_hres_only --dataset bowl

In [ ]:
!python3 evaluate.py --model htan_1_hres_only --dataset bowl

## 3.4 Evaluate All Models

In [ ]:
!python3 evaluate.py --model all --dataset bowl

## 3.5 Results Table

In [ ]:
RESULTS_DIR = Path("/opt/dlami/nvme/HTAN/results/bowl")
SAVES_ROOT  = Path("/opt/dlami/nvme/HTAN/saves")

# Paper numbers — Table IV
PAPER_RESULTS = {
    "U-Net†":          {"dice": 75.73, "iou": 91.03, "acc": None,  "rec": None,  "pre": None},
    "FANet†":          {"dice": 81.03, "iou": 71.08, "acc": 95.59, "rec": 80.62, "pre": 82.31},
    "Channel-UNet†":   {"dice": 87.55, "iou": 79.75, "acc": 96.27, "rec": 90.70, "pre": 87.86},
    "ResUNet†":        {"dice": 89.91, "iou": 82.44, "acc": 97.05, "rec": 90.00, "pre": 90.84},
    "Attention U-Net†":{"dice": 90.83, "iou": 91.03, "acc": None,  "rec": None,  "pre": 91.61},
    "DoubleU-Net†":    {"dice": 91.33, "iou": 84.07, "acc": None,  "rec": 64.07, "pre": 94.96},
    "TransAttUNet_R†": {"dice": 91.62, "iou": 84.98, "acc": 97.46, "rec": 91.85, "pre": 91.93},
}

MODEL_LABELS = {
    "transattunet":     "TransAttUNet_R*",
    "htan_1_n2":        "HTAN_1 n=2 (Ours)",
    "htan_1_n4":        "HTAN_1 n=4 (Ours)",
    "htan_2_n2":        "HTAN_2 n=2 (Ours)",
    "htan_1_hres_only": "HTAN_1 Hres-only (Ours)",
}

OUR_RESULTS = {}
for key, label in MODEL_LABELS.items():
    path = RESULTS_DIR / f"{key}.json"
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        OUR_RESULTS[label] = {k: data[k] for k in ["dice","iou","acc","rec","pre"]}
    else:
        print(f"Not trained yet: {key}")

def fmt(v):
    return f"{v:.2f}" if v is not None else "—"

ALL = {**PAPER_RESULTS, **OUR_RESULTS}
print(f"\n{'Method':<26} {'Dice':>6} {'IoU':>6} {'ACC':>6} {'REC':>6} {'PRE':>6}")
print("-" * 58)
for name, m in ALL.items():
    print(f"{name:<26} {fmt(m['dice']):>6} {fmt(m['iou']):>6} "
          f"{fmt(m['acc']):>6} {fmt(m['rec']):>6} {fmt(m['pre']):>6}")

## 3.6 Training Curves

In [ ]:
def load_history(model_name, dataset="bowl"):
    path = SAVES_ROOT / f"{model_name}_{dataset}" / "resume_checkpoint.pth"
    if not path.exists():
        return None
    ckpt = torch.load(path, map_location="cpu")
    return ckpt.get("history", None)

MODELS = list(MODEL_LABELS.keys())
histories = {m: load_history(m) for m in MODELS}

def plot_metric(histories, metric, title):
    plt.figure(figsize=(10, 5))
    for name, h in histories.items():
        if h and metric in h:
            plt.plot(h[metric], label=name)
    plt.xlabel("Epoch")
    plt.ylabel(metric.capitalize())
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_metric(histories, "dice",       "3.6.1 Validation Dice — Bowl")
plot_metric(histories, "train_loss", "3.6.2 Training Loss — Bowl")

## 3.7 Visual Predictions

In [ ]:
from datasets.bowl_dataset import get_loaders

_, val_loader, _ = get_loaders(img_size=256, batch_size=4)
imgs, masks = next(iter(val_loader))
imgs_gpu    = imgs.to(DEVICE)

tan  = TransAttUNet_R()
htan = HTAN_2(expansion_n=2, img_size=256)

def load_best(model, model_name, dataset="bowl"):
    path = SAVES_ROOT / f"{model_name}_{dataset}" / "best_model.pth"
    if not path.exists():
        print(f"No checkpoint: {model_name}_{dataset}")
        return None
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    return model.to(DEVICE).eval()

tan  = load_best(tan,  "transattunet")
htan = load_best(htan, "htan_2_n2")

with torch.no_grad():
    pred_tan  = (torch.sigmoid(tan(imgs_gpu))  > 0.5).cpu() if tan  else None
    pred_htan = (torch.sigmoid(htan(imgs_gpu)) > 0.5).cpu() if htan else None

def denorm(t):
    return (t * 0.5 + 0.5).clamp(0, 1)

n_show = min(4, imgs.shape[0])
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4*n_show))
fig.suptitle("3.7 Visual Predictions — Bowl", fontsize=14, fontweight="bold")

for col, title in enumerate(["Input", "Ground Truth", "TransAttUNet_R", "HTAN_2 n=2"]):
    axes[0, col].set_title(title, fontweight="bold")

for i in range(n_show):
    axes[i,0].imshow(denorm(imgs[i]).permute(1,2,0).numpy())
    axes[i,1].imshow(masks[i,0].numpy(), cmap="gray")
    axes[i,2].imshow(pred_tan[i,0].numpy()  if pred_tan  is not None else np.zeros((256,256)), cmap="gray")
    axes[i,3].imshow(pred_htan[i,0].numpy() if pred_htan is not None else np.zeros((256,256)), cmap="gray")
    for ax in axes[i]: ax.axis("off")

plt.tight_layout()
plt.show()